# MC3 model test v6: sample set + a harder practice set

Runs the real MC3 code with a small model (Qwen3-VL-4B) on a free GPU, on the 10 sample questions AND on a harder set of 32 questions
(look-alike documents, big logs, spreadsheets, images with meaningless file names, two-step lookups, unanswerable questions).

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Then run the cells one at a time, top to bottom.
**Keep this tab open and active.** The model download (cell 3) can take from a few minutes up to about 1.5 hours on a slow day.
If Colab resets, start again from cell 2.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Upload the kit
Pick **mc3-kit-v6.zip**. The last line must say `kit v6 OK`.

In [ ]:
!rm -f /content/*.zip
from google.colab import files
up = files.upload()
name = list(up.keys())[0]
print("using:", name)
!rm -rf /content/kit && mkdir -p /content/kit && unzip -q -o "/content/{name}" -d /content/kit
%cd /content/kit/mc3-starter-kit
!grep -q 'approved document' starter/app/rag/llm.py && echo "kit v6 OK" || echo "WRONG ZIP: this is an old kit, upload mc3-kit-v6.zip"

## 3. Set up: packages, model, and the two indexes
The slow cell. It ends with the model server log saying `ready`. If it fails, send me the last lines.

In [ ]:
%cd /content/kit/mc3-starter-kit
!pip install -q -U "transformers>=4.57" pypdf huggingface_hub
!sh setup.sh
!HF_HUB_OFFLINE=0 TRANSFORMERS_OFFLINE=0 python3 -c "from huggingface_hub import snapshot_download; snapshot_download('Qwen/Qwen3-VL-4B-Instruct', local_dir='/content/model', allow_patterns=['*.json','*.safetensors','*.txt','*.model','*.jinja']); print('model downloaded')"
import os
os.environ.update({
    "MC3_MODEL_DIR": "/content/model", "MC3_DTYPE": "float16", "MC3_LLM_SOCKET": "/tmp/mc3.sock",
    "MC2_OUTPUT_DIR": "/content/out", "MC3_DEBUG": "1",
    "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
})
print("--- index: sample corpus ---")
!MC3_INDEX_DIR=/content/idx_sample python3 starter/app/app.py --index mc3-corpus
print("--- index: hard corpus ---")
!MC3_INDEX_DIR=/content/idx_hard python3 starter/app/app.py --index mc3-corpus-hard
!tail -n 5 /tmp/mc3_model_server.log

## 4. Sample set (10 questions)
Last time this scored 10/10. Each line: PASS or FAIL, time, answer, citations.

In [ ]:
!MC3_INDEX_DIR=/content/idx_sample python3 score_samples.py --local mc3-corpus

## 5. What did the model read from the 3 hard-set images?
They have meaningless file names, so the model's reading is the only way to find them.
Expected: a nameplate (serial HR-2207-0418), a torque table (J4 = 28), a connector diagram (BRAKE_REL on pin 14).

In [ ]:
import pickle
idx = pickle.load(open("/content/idx_hard/index.pkl", "rb"))
for d in idx["docs"]:
    if d["kind"] == "image":
        print(d["file"]); print(d["text"][:600]); print("-" * 50)

## 6. Hard set (32 questions)
Takes a few minutes. One question is skipped on purpose (it needs a non-admin user). For each FAIL it prints the model's raw reply.

In [ ]:
!MC3_INDEX_DIR=/content/idx_hard python3 score_samples.py --local mc3-corpus-hard --questions hard-questions.json

## 7. Clean up
Stops the model server and frees the GPU.

In [ ]:
!pkill -f "[m]odel_server.py" 

## What to send back
The output of **cells 5 and 6** (screenshots or copied text) and, if anything failed, of the cell that failed.
The more FAIL lines with their `process said: RAW MODEL REPLY` text, the more I can fix.